# Vector Search

In [1]:
from sentence_transformers import SentenceTransformer
from gitsource import GithubRepositoryDataReader, chunk_documents
from openai import OpenAI
from dotenv import load_dotenv
from minsearch import VectorSearch, Index
from pydantic import BaseModel, Field
from typing import Literal
import json
import os

In [2]:
model = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
load_dotenv()
api_key = os.environ.get("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

## Embeddings Tutorial

In [4]:
q1 = 'install Evidently locally'
q2 = 'how can I set up evidently in my projects?'

In [5]:
v1 = model.encode(q1)
v1[:20]

array([ 0.07184876, -0.02879697,  0.06123491, -0.01810756,  0.125331  ,
       -0.04134635, -0.06354985, -0.08986842, -0.04142815, -0.04465349,
        0.06421644,  0.02077091,  0.00219161,  0.00508833, -0.00833962,
       -0.03770015,  0.03059413, -0.04844621,  0.06848753, -0.01030121],
      dtype=float32)

In [6]:
v1.shape

(384,)

In [7]:
v2 = model.encode(q2)

In [8]:
v1.dot(v2)

np.float32(0.43477935)

In [9]:
q3 = 'How fast can an African sparrow fly carrying a coconut'
v3 = model.encode(q3)

In [10]:
v1.dot(v3)

np.float32(0.0057888404)

## Creating Document Embeddings

In [11]:
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"}
)

files = reader.read()
parsed_docs = [parsed.parse() for parsed in files]

chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

In [12]:
texts = []

for doc in chunked_docs:
    title = doc.get('title', '')
    description = doc.get('description', '')
    content = doc.get('content', '')
    text = title + " " + description + " " + content
    texts.append(text.strip())

In [13]:
embeddings = model.encode(texts, show_progress_bar=True)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

In [13]:
embeddings.shape

(385, 384)

## Vector Search with Minsearch

In [36]:
scores = embeddings.dot(v1)
scores[:15]

array([ 0.07959675,  0.12935512,  0.18664968,  0.26054296,  0.08233505,
       -0.02807793, -0.07214396, -0.05210196, -0.01589229,  0.00971265,
       -0.05274098, -0.10643858,  0.04062269,  0.06344485, -0.04217627],
      dtype=float32)

In [37]:
vindex = VectorSearch(
    keyword_fields=['filename']
)
vindex.fit(embeddings, chunked_docs)

In [38]:
def vector_search(query, num_results=5):
    vector = model.encode(query)
    return vindex.search(vector, num_results=num_results)

## RAG with Vector Search

In [39]:
class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [40]:
instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

In [41]:
def structured_llm(client, user_prompt, output_type, system_instructions=None, model="gpt-4o-mini"):
    messages = []

    if system_instructions:
        messages.append({
            "role":"system",
            "content":system_instructions
        })
    
    messages.append({
        "role":"user",
        "content":user_prompt
    })

    response = client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed

In [42]:
def augment(question, search):
    context = json.dumps(search, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

def rag_vector(query, output_format=RAGResponse):
    search_results = vector_search(query)
    prompt = augment(query, search_results)
    return structured_llm(client=client, user_prompt=prompt, output_type=RAGResponse, system_instructions=instructions)

In [43]:
question = "How do I use Evidently in my project?"
answer = rag_vector(question)

print(f"Query: {question}")
print(f"found_answer: {answer.found_answer}")
print(f"confidence: {answer.confidence}")
print(f"answer: {answer.answer}")

Query: How do I use Evidently in my project?
found_answer: True
confidence: 0.95
answer: To use Evidently in your project, follow these steps:

1. **Install Evidently**: You can install the Evidently Python library using pip or conda:
   - Using pip:
     ```bash
     pip install evidently
     ```
   - Using conda:
     ```bash
     conda install -c conda-forge evidently
     ```

2. **Launch the UI service**: Run the Evidently UI service with the appropriate command in your terminal:
   - If your workspace is local, navigate to the directory and run:
     ```bash
     evidently ui
     ```
   - For a different workspace location, specify the path:
     ```bash
     evidently ui --workspace ./your_workspace
     ```
   - To run on a specific port (if needed), use:
     ```bash
     evidently ui --workspace ./your_workspace --port 8080
     ```

3. **View the Interface**: Open a web browser and go to `http://localhost:8000` or the specified port to view the Evidently interface.

4. **E

## Hybrid Search

In [32]:
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)

index.fit(chunked_docs)

def search(query):
    return index.search(
        query=query,
        num_results=5,
    )

In [33]:
def hybrid_search(query):
    vector_results = vector_search(query)
    text_results = search(query)
    return vector_results + text_results

In [35]:
def rag_hybrid(query, output_format=RAGResponse):
    search_results = hybrid_search(query)
    prompt = augment(query, search_results)
    return structured_llm(client=client, user_prompt=prompt, output_type=output_format, system_instructions=instructions)

In [44]:
question = "How do I install Ubuntu on my computer?"
answer = rag_hybrid(question)

print(f"Query: {question}")
print(f"found_answer: {answer.found_answer}")
print(f"confidence: {answer.confidence}")
print(f"answer: {answer.answer}")

Query: How do I install Ubuntu on my computer?
found_answer: False
confidence: 0.0
answer: The provided context does not contain information on how to install Ubuntu on your computer. Please refer to the official Ubuntu documentation or installation guides for the steps to install Ubuntu.
